In [1]:
import pandas as pd
import json

from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.svm import SVR,NuSVR

import pandas as pd
from helpers.modeling import (
    identify_column_types,
    create_preprocessor,
    evaluate_model,
)

from tqdm import tqdm


with open('results.json', 'r') as f:
    results = json.load(f)

In [2]:
df = pd.read_csv("../Datasets/processed/UHPC_dataset/semantic_recoding_features_50_with_publications.csv")
df.head()
df['paper_reference'].nunique()
df['paper_reference'].value_counts().describe()

count    165.000000
mean      12.563636
std       15.086454
min        1.000000
25%        4.000000
50%        8.000000
75%       16.000000
max      112.000000
Name: count, dtype: float64

In [3]:
def run_pipeline(model_cls, model_key, kernel_kwargs=None):
    params = results["best_params"]["best_params_publications_included"][model_key]
    pipeline = Pipeline([('preprocessor', preprocessor),
                          ('model', model_cls(**(kernel_kwargs or {})))])
    pipeline.set_params(**params)
    pipeline.fit(X_train, y_train)

    train_metrics = evaluate_model(y_train, pipeline.predict(X_train))
    test_metrics = evaluate_model(y_test, pipeline.predict(X_test))
    return pipeline, train_metrics, test_metrics

In [4]:
X = df.drop(columns=["cs_28d", "paper_reference"])
y = df["cs_28d"]

pub_col = "paper_reference" 

numerical_cols, one_hot_columns, k_fold_columns = identify_column_types(X)

preprocessor = create_preprocessor(numerical_cols, one_hot_columns, k_fold_columns, 
                                   handle_unknown='ignore') 


In [5]:
total_rows = 0
for pub, group in df.groupby('paper_reference'):
    print(f"{pub}: {len(group)} rows")
    total_rows += len(group)
    
print(total_rows)

Ref-1-data: 16 rows
Ref-10-Research: 7 rows
Ref-100-Research: 8 rows
Ref-101-Research: 28 rows
Ref-102-Research: 16 rows
Ref-103-Research: 10 rows
Ref-104-Research: 8 rows
Ref-105-Research: 7 rows
Ref-106-Research: 8 rows
Ref-107-Research: 16 rows
Ref-108-Research: 3 rows
Ref-109-Research: 2 rows
Ref-11-Research : 4 rows
Ref-110-Research: 18 rows
Ref-111-Research: 2 rows
Ref-113-Research: 5 rows
Ref-114-Research: 5 rows
Ref-115-Research: 5 rows
Ref-116-Research: 36 rows
Ref-117-Research: 5 rows
Ref-118-Research: 5 rows
Ref-119-Research: 1 rows
Ref-12-Research: 4 rows
Ref-120-Research: 4 rows
Ref-121-Research: 80 rows
Ref-122-Research: 18 rows
Ref-123-Research: 11 rows
Ref-124-Research: 10 rows
Ref-125-Research: 16 rows
Ref-126-Research: 5 rows
Ref-127-Research: 3 rows
Ref-128-Research: 3 rows
Ref-129-Research: 1 rows
Ref-13-Research: 27 rows
Ref-130-Research: 6 rows
Ref-132-Research: 12 rows
Ref-133-Research: 2 rows
Ref-134-Research: 20 rows
Ref-135-Research: 43 rows
Ref-136-Research: 

In [6]:
model_configs = [
    (KNeighborsRegressor, 'knn', None),
    (SVR, 'svr', {'kernel': 'rbf'}),
    (NuSVR, 'nusvr', {'kernel': 'rbf'}),
]

groups = list(df.groupby(pub_col))
threshold = 50
lopo_results = []
lopo_predictions = []  
total_rows = 0

for pub_id, group_df in tqdm(groups, total=len(groups)):
    if len(group_df) < threshold:   #threshold
        continue
    total_rows += len(group_df)

    train_mask = df[pub_col] != pub_id
    test_mask  = df[pub_col] == pub_id

    X_train, y_train = X[train_mask], y[train_mask]
    X_test,  y_test  = X[test_mask],  y[test_mask]

    for model_cls, model_key, kwargs in model_configs:
        pipeline, train_metrics, test_metrics = run_pipeline(model_cls, model_key, kwargs)
        test_metrics.update({'publication': pub_id, 'model': model_key, 'train_RMSE': train_metrics['RMSE']})
        lopo_results.append(test_metrics)

        y_pred = pipeline.predict(X_test)
        for true_val, pred_val, idx in zip(y_test.values, y_pred, y_test.index):
            lopo_predictions.append({'index': idx, 'publication': pub_id, 'model': model_key,
                                       'y_true': true_val, 'y_pred': pred_val})

lopo_df = pd.DataFrame(lopo_results)
lopo_pred_df = pd.DataFrame(lopo_predictions)
print(f"total rows after thresholding ({threshold}) : {total_rows}")

  0%|          | 0/165 [00:00<?, ?it/s]c:\Users\shoai\anaconda3\envs\aicome\Lib\site-packages\threadpoolctl.py:1214: RuntimeWarning: 
Found Intel OpenMP ('libiomp') and LLVM OpenMP ('libomp') loaded at
the same time. Both libraries are known to be incompatible and this
can cause random crashes or deadlocks on Linux when loaded in the
same Python program.
Using threadpoolctl may cause crashes or deadlocks. For more
information and possible workarounds, please see
    https://github.com/joblib/threadpoolctl/blob/master/multiple_openmp.md

  warnings.warn(msg, RuntimeWarning)
100%|██████████| 165/165 [00:18<00:00,  9.01it/s]

total rows after thresholding (50) : 452


In [7]:
print(f"unique values after threshold  {threshold}: {lopo_df['publication'].nunique()}")


unique values after threshold  50: 6


In [8]:
# Pooled LOPO summary:
# Every row in the dataset was held out exactly once (by the model that never
# saw its publication during training). This pools ALL those held-out
# predictions together (across all 165 eligible publications) and computes
# ONE overall RMSE/MAE/R2/Correlation/Mean_Residual per model - as if it were
# a single big "unseen publication" test set.

pooled_summary = []
for model_key, group in lopo_pred_df.groupby('model'):
    m = evaluate_model(group['y_true'], group['y_pred'])
    m['model'] = model_key
    m['N_total'] = len(group)
    pooled_summary.append(m)

pooled_lopo_df = pd.DataFrame(pooled_summary)
pooled_lopo_df

,RMSE,MAE,R2,Correlation,Mean_Residual,N,model,N_total
0,24.959460,19.400031,0.408587,0.657710,4.572653,452,knn,452
1,31.956466,23.487133,0.030522,0.413568,8.545990,452,nusvr,452
2,32.232125,23.933210,0.013724,0.429071,9.696209,452,svr,452


In [9]:
lopo_pred_df['residual'] = lopo_pred_df['y_true'] - lopo_pred_df['y_pred']

residual_pivot = lopo_pred_df.pivot(index='index', columns='model', values='residual')
residual_pivot['mean_abs_residual'] = residual_pivot[['knn', 'svr', 'nusvr']].abs().mean(axis=1)
residual_pivot['same_direction'] = residual_pivot[['knn', 'svr', 'nusvr']].apply(
    lambda r: (r > 0).all() or (r < 0).all(), axis=1
)

worst_rows = residual_pivot.sort_values('mean_abs_residual', ascending=False).head(10)
worst_rows

model,knn,nusvr,svr,mean_abs_residual,same_direction
index,,,,,
547,32.585927,106.403254,108.587793,82.525658,True
548,30.593830,104.152834,107.423001,80.723222,True
549,25.600467,98.079815,101.110202,74.930161,True
527,35.741605,92.102605,94.289253,74.044488,True
551,41.305433,88.263744,89.427585,72.998921,True
546,25.576355,95.985302,95.717727,72.426462,True
528,33.721279,89.045965,92.655813,71.807685,True
531,59.044066,77.238307,78.397487,71.559953,True
507,47.778648,80.543295,82.802744,70.374896,True


In [10]:
worst_idx = worst_rows.head(10).index 

df.loc[worst_idx].T

index,547,548,549,527,551,546,528,531,507,1040
cement,850.0,850.0,850.0,850.0,850.0,850.0,850.0,850.0,850.0,960.0
cement_type,OPC_53,OPC_53,OPC_53,OPC_53,OPC_53,OPC_53,OPC_53,OPC_53,OPC_53,OPC_42.5
silica_fume,260.0,260.0,260.0,260.0,260.0,260.0,260.0,260.0,260.0,288.0
fly_ash,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
fly_ash_type,not_applicable,not_applicable,not_applicable,not_applicable,not_applicable,not_applicable,not_applicable,not_applicable,not_applicable,not_applicable
limestone_powder,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
quartz_powder,212.0,212.0,212.0,212.0,212.0,212.0,212.0,212.0,212.0,0.0
glass_powder,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
rice_husk_ash,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
metakaolin,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [11]:
lopo_df.to_csv("results/lopo_results.csv", index=False)

In [12]:
summary = lopo_df.groupby('model')[['RMSE', 'MAE', 'R2', 'Correlation', 'Mean_Residual']].agg(['mean', 'median', 'std'])
summary

RMSE                              MAE                        \
            mean     median        std       mean     median        std   
model                                                                     
knn    24.339737  23.058590   7.355604  19.631896  18.160654   6.151728   
nusvr  29.301040  24.116781  16.217406  24.302984  18.148558  15.453511   
svr    29.683776  24.050846  15.999936  24.744263  18.232848  15.115886   

             R2                     Correlation                      \
           mean    median       std        mean    median       std   
model                                                                 
knn   -0.025547 -0.044920  0.373457    0.518244  0.574293  0.249125   
nusvr -0.606590 -0.368307  1.376932    0.458462  0.529664  0.420894   
svr   -0.647256 -0.480228  1.365805    0.462541  0.517001  0.402923   

      Mean_Residual                       
               mean    median        std  
model                                     
knn        5.579821  3.509307  12.385253  
nusvr      9.296539 -0.404218  20.422747  
svr       10.541309  2.447384  20.384122

In [13]:
per_pub = lopo_df[['publication', 'model', 'N', 'RMSE', 'MAE', 'R2', 'Correlation', 'Mean_Residual']].sort_values(['model', 'publication'])
per_pub.to_csv("results/lopo_per_publication.csv", index=False)

In [14]:
#worst publivations
worst = lopo_df.sort_values('R2').groupby('model').head(10)[['model', 'publication', 'N', 'RMSE', 'R2', 'Correlation', 'Mean_Residual']]
worst

,model,publication,N,RMSE,R2,Correlation,Mean_Residual
13,svr,Ref-48-Research,72,58.011052,-3.207993,-0.124005,38.114328
14,nusvr,Ref-48-Research,72,57.874921,-3.188267,-0.161478,37.271917
5,nusvr,Ref-139-Research,51,26.377567,-0.783407,0.137884,-10.644036
4,svr,Ref-139-Research,51,26.031189,-0.736877,0.169541,-10.457723
16,svr,Ref-85-Research,64,38.122195,-0.573316,0.941294,34.006291
17,nusvr,Ref-85-Research,64,37.682890,-0.537265,0.935235,32.971646
15,knn,Ref-85-Research,64,36.846205,-0.469758,0.802487,26.163089
7,svr,Ref-141-Research,73,14.706298,-0.387139,0.341609,3.640992
6,knn,Ref-141-Research,73,14.625932,-0.372019,0.296419,8.388079
8,nusvr,Ref-141-Research,73,13.674652,-0.199349,0.359442,-1.981862


In [15]:
lopo_df['direction'] = lopo_df['Mean_Residual'].apply(lambda x: 'under-predicted' if x > 0 else 'over-predicted')
direction_summary = lopo_df.groupby(['model', 'direction']).size().unstack(fill_value=0)
direction_summary

direction,over-predicted,under-predicted
model,,
knn,3,3
nusvr,3,3
svr,2,4
